# Homomorfik Matris–Vektör Çarpımı — Diagonal (Halevi–Shoup) Yöntemi





## 0. Kurulum

In [1]:
# uv ile Python 3.10 ortamı + Ubuntu 22.04 için OpenFHE wheel'i (Colab'a uyan tek kombinasyon)
!pip install -q uv
!uv python install 3.10
!uv venv /content/fhe310 --python 3.10
!uv pip install --python /content/fhe310/bin/python "openfhe==1.5.1.0.22.4" numpy
!/content/fhe310/bin/python -c "import openfhe, sys; print('OpenFHE hazir, Python', sys.version.split()[0])"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 29.8 MB/s eta 0:00:00
Installed Python 3.10.20 in 1.16s
 + cpython-3.10.20-linux-x86_64-gnu (python3.10)
Using CPython 3.10.20
Creating virtual environment at: fhe310
Activate with: source fhe310/bin/activate
Using Python 3.10.20 environment at: fhe310
Resolved 2 packages in 665ms
Prepared 2 packages in 1.21s
Installed 2 packages in 17ms
 + numpy==2.2.6
 + openfhe==1.5.1.0.22.4
OpenFHE hazir, Python 3.10.20


## 1. FHE kodu



In [2]:
%%writefile /content/matvec.py
# Diagonal (Halevi-Shoup) homomorphic matrix-vector product -- OpenFHE / CKKS (CPU)
import time
import numpy as np
from openfhe import CCParamsCKKSRNS, GenCryptoContext, PKESchemeFeature

n = 4                                     # matris boyutu (2, 8, 16 ... ikinin kuvveti)
RUN_TABLE = True                          # ölçekleme tablosunu da çalıştır

rng = np.random.default_rng(42)
M = rng.integers(1, 13, size=(n, n)).astype(float)
v = rng.integers(1, 13, size=n).astype(float)
reference = M @ v

def diagonalize(M):
    n = M.shape[0]
    return {k: np.array([M[i, (i + k) % n] for i in range(n)]) for k in range(n)}

def matvec(n, M, v):
    diags = diagonalize(M)
    p = CCParamsCKKSRNS()
    p.SetMultiplicativeDepth(1); p.SetScalingModSize(45); p.SetFirstModSize(60)
    cc = GenCryptoContext(p)
    for f in (PKESchemeFeature.PKE, PKESchemeFeature.LEVELEDSHE, PKESchemeFeature.KEYSWITCH):
        cc.Enable(f)
    slots = cc.GetRingDimension() // 2; reps = slots // n
    keys = cc.KeyGen(); cc.EvalMultKeyGen(keys.secretKey)
    cc.EvalRotateKeyGen(keys.secretKey, list(range(1, n)))
    tile = lambda a: np.tile(a, reps).tolist()
    dpts = {k: cc.MakeCKKSPackedPlaintext(tile(diags[k])) for k in diags}
    ctv = cc.Encrypt(keys.publicKey, cc.MakeCKKSPackedPlaintext(tile(v)))
    ops = {"rot": 0, "pmult": 0, "add": 0}
    t0 = time.perf_counter(); acc = None
    for k in range(n):
        cr = cc.EvalRotate(ctv, k) if k else ctv
        if k: ops["rot"] += 1
        part = cc.EvalMult(cr, dpts[k]); ops["pmult"] += 1
        if acc is None: acc = part
        else: acc = cc.EvalAdd(acc, part); ops["add"] += 1
    dt = time.perf_counter() - t0
    out = cc.Decrypt(keys.secretKey, acc); out.SetLength(n)
    return np.array(out.GetRealPackedValue()), ops, dt, cc.GetRingDimension()

he, ops, dt, ringdim = matvec(n, M, v)
err = float(np.abs(he - reference).mean())
prec = -np.log2(err) if err > 0 else float("inf")

print("=" * 60)
print(f"Diagonal matris-vektor carpimi (OpenFHE/CKKS) | n={n}, ring={ringdim}")
print("=" * 60)
print("numpy   M@v =", reference)
print("OpenFHE M@v =", np.round(he, 6))
print("eslesme     :", np.allclose(he, reference, atol=1e-4), f"| ort.hata {err:.2e} (~{prec:.1f} bit)")
print(f"rotasyon={ops['rot']} (n-1) | plaintext-carpim={ops['pmult']} (n) | toplama={ops['add']} | derinlik=1")
print(f"matvec suresi: {dt:.4f} s")
print("=" * 60)

if RUN_TABLE:
    print("\nOlcekleme tablosu:")
    print(f"{'n':>4} {'rot':>5} {'pmult':>6} {'matvec(s)':>11} {'mean_err':>11} {'ok':>6}")
    for nn in [2, 4, 8, 16, 32, 64, 128]:
        r = np.random.default_rng(42)
        Mx = r.integers(1,13,size=(nn,nn)).astype(float); vx = r.integers(1,13,size=nn).astype(float)
        hx, o, d, _ = matvec(nn, Mx, vx)
        e = np.abs(hx - Mx@vx).mean()
        print(f"{nn:>4} {o['rot']:>5} {o['pmult']:>6} {d:>11.4f} {e:>11.2e} {str(np.allclose(hx,Mx@vx,atol=1e-4)):>6}")


Writing /content/matvec.py


## 2. Çalıştır


In [3]:
!/content/fhe310/bin/python /content/matvec.py

Diagonal matris-vektor carpimi (OpenFHE/CKKS) | n=4, ring=16384
numpy   M@v = [158. 140. 174. 242.]
OpenFHE M@v = [158. 140. 174. 242.]
eslesme     : True | ort.hata 1.30e-09 (~29.5 bit)
rotasyon=3 (n-1) | plaintext-carpim=4 (n) | toplama=3 | derinlik=1
matvec suresi: 0.0474 s

Olcekleme tablosu:
   n   rot  pmult   matvec(s)    mean_err     ok
   2     1      2      0.0179    1.75e-09   True
   4     3      4      0.0523    8.61e-10   True
   8     7      8      0.1044    2.62e-09   True
  16    15     16      0.3539    3.20e-09   True
  32    31     32      1.3514    3.95e-09   True
  64    63     64      0.9329    5.13e-09   True
 128   127    128      1.9293    6.25e-09   True
